# State-Space Geometry of V1 Population Activity via Pseudopopulation PCA

This notebook investigates the average state-space geometry of neural population responses to treadmill stimuli in Layer 2/3 and Layer 5 sessions using **Pseudopopulation PCA**.

### Method: Pseudopopulation PCA
Because stimulus conditions (expected Running Speed and expected Optic Flow) are identical across sessions, we can combine different recordings into a single giant population matrix:
1. Load treadmill trials with default synchronization parameters (cutting the initial acceleration ramp to steady-state locomotion).
2. Keep only closed-loop trials and map the expected OF and expected RS values.
3. Bin the 2D behavioral space (expected RS x expected OF) to yield a $(30, C_s)$ response matrix for each session (where $C_s$ is the cell count).
4. Standardize (Z-score) each cell across conditions within its session to ensure comparable variances.
5. Concatenate session matrices column-wise for Layer 2/3 (nominal depth $\le$ 350 µm) and Layer 5 (nominal depth > 350 µm) to build two massive pseudopopulations.
6. Perform Principal Component Analysis (PCA) on each layer's pseudopopulation matrix and project the 30 conditions onto the first two PCs.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
import matplotlib.font_manager as fm

import flexiznam as flz
from cottage_analysis.analysis import treadmill
from cottage_analysis.summary_analysis import get_session_list

In [ ]:
# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording"
}

In [ ]:
def load_session_trials(sess, project, flexilims_session):
    """
    Loads the trial dataframe for a session using default synchronization parameters.
    Since default sync uses acceleration_time=0.13, it cuts the treadmill acceleration periods.
    """
    filter_datasets = {"anatomical_only": 3, "ast_neuropil": False}
    annotated_filter = dict(filter_datasets)
    annotated_filter["annotated"] = True
    
    suite2p_datasets = flz.get_datasets(
        origin_name=sess,
        dataset_type="suite2p_rois",
        project_id=project,
        flexilims_session=flexilims_session,
        return_dataseries=False,
        filter_datasets={"anatomical_only": 3},
    )
    if not suite2p_datasets:
        raise FileNotFoundError(f"No suite2p_rois dataset found for {sess}")
    frame_rate = suite2p_datasets[0].extra_attributes["fs"]
    
    try:
        vs_df, trials_df = treadmill.sync_all_recordings(
            session_name=sess,
            flexilims_session=flexilims_session,
            project=project,
            filter_datasets=annotated_filter,
            recording_type="two_photon",
            photodiode_protocol=5,
            return_volumes=True,
        )
    except FileNotFoundError:
        vs_df, trials_df = treadmill.sync_all_recordings(
            session_name=sess,
            flexilims_session=flexilims_session,
            project=project,
            filter_datasets=filter_datasets,
            recording_type="two_photon",
            photodiode_protocol=5,
            return_volumes=True,
        )
        
    return trials_df, frame_rate

In [ ]:
# Load all sessions across projects and group by layer
max_abs_rs2motor_diff_ratio = 0.3
projects = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
layer_cells = {"Layer 2/3": [], "Layer 5": []}

for project in projects:
    flm_sess = flz.get_flexilims_session(project_id=project)
    sessions = get_session_list.get_motor_session_list(flm_sess)
    project_sessions = flz.get_entities("session", flexilims_session=flm_sess)
    
    for sess in sessions:
        if sess in SESSIONS_TO_EXCLUDE:
            continue
        if sess not in project_sessions.index:
            continue
            
        nominal_depth = project_sessions.loc[sess, "nominal_depth"]
        if isinstance(nominal_depth, (list, np.ndarray, pd.Series)):
            nominal_depth = np.mean(nominal_depth)
        layer = "Layer 2/3" if nominal_depth <= 350 else "Layer 5"
        
        try:
            print(f"Processing {sess} ({layer})...")
            trials_df, fs = load_session_trials(sess, project, flm_sess)
            
            # Keep closed loop
            trials_df = trials_df[trials_df.closed_loop == 1].copy()
            if len(trials_df) == 0:
                continue
                
            trials_df["expected_OF"] = trials_df.expected_optic_flow_stim.map(np.nanmedian)
            trials_df["expected_RS"] = trials_df.MotorSpeed_stim.map(np.nanmedian)
            trials_df = trials_df.dropna(subset=["expected_OF", "expected_RS"])
            
            # Compute average population vector for each condition
            bin_responses = {}
            for (speed, of), group in trials_df.groupby(["expected_RS", "expected_OF"]):
                valid_dffs = []
                for _, trial in group.iterrows():
                    dff = trial["dff_stim"]
                    if not isinstance(dff, np.ndarray):
                        continue
                    
                    # Create boolean mask for static frames
                    ratio = trial["max_abs_rs2motor_diff_ratio_stim"]
                    ok = ratio < max_abs_rs2motor_diff_ratio
                    
                    # Filter frames (rows) for this trial
                    if np.any(ok):
                        valid_dffs.append(dff[ok, :])
                if len(valid_dffs) == 0:
                    continue
                stacked = np.vstack(valid_dffs)
                mean_vector = np.nanmean(stacked, axis=0)
                bin_responses[(speed, of)] = mean_vector
                
            layer_cells[layer].append({
                "session": sess,
                "nominal_depth": nominal_depth,
                "responses": bin_responses
            })
        except Exception as e:
            print(f"Error processing {sess}: {e}")

In [ ]:
1+1

In [ ]:
# For each session, plot variance explain vs dimension color coded by nominal depth
sess_data_list = layer_cells["Layer 2/3"]
for sd in sess_data_list:
    break

In [ ]:
sd.shape

In [ ]:
# Define the aligned grid conditions
all_speeds = [3.8125, 7.0, 15.0, 30.0, 60.0]
all_ofs = [1.0, 4.0, 16.0, 64.0, 256.0, 1024.0]
conditions = [(s, o) for s in all_speeds for o in all_ofs]

def build_pca_pseudopopulation(layer):
    sess_data_list = layer_cells[layer]
    
    # Keep sessions that contain responses for all 30 conditions
    valid_sess_list = []
    for sd in sess_data_list:
        if len(sd["responses"]) == 30:
            valid_sess_list.append(sd)
            
    print(f"[{layer}] Found {len(valid_sess_list)} sessions with all 30 conditions.")
    if not valid_sess_list:
        raise ValueError(f"No valid sessions with 30 conditions for {layer}")
        
    X_parts = []
    for sd in valid_sess_list:
        X_sess = []
        for cond in conditions:
            X_sess.append(sd["responses"][cond])
        X_sess = np.array(X_sess) # 30 x C_s
        
        # Standardize cell responses across conditions within session
        X_sess_norm = (X_sess - np.mean(X_sess, axis=0)) / (np.std(X_sess, axis=0) + 1e-6)
        X_parts.append(X_sess_norm)
        
    X_total = np.hstack(X_parts) # 30 x C_total
    
    # Drop columns containing NaNs
    nan_cols = np.isnan(X_total).any(axis=0)
    if np.any(nan_cols):
        print(f"[{layer}] Dropping {np.sum(nan_cols)} columns with NaN values.")
        X_total = X_total[:, ~nan_cols]
        
    # Perform PCA
    pca = PCA(n_components=None)
    X_proj = pca.fit_transform(X_total)
    var_explained = pca.explained_variance_ratio_
    
    # Build projection dataframe
    proj_df = pd.DataFrame([{"expected_RS": s, "expected_OF": o} for s, o in conditions])
    proj_df["PC1"] = X_proj[:, 0]
    proj_df["PC2"] = X_proj[:, 1]
    proj_df["expected_depth"] = proj_df["expected_RS"] / proj_df["expected_OF"]
    proj_df["log_expected_depth"] = np.log2(proj_df["expected_depth"])
    
    return proj_df, var_explained, X_total.shape[1], len(valid_sess_list)

In [ ]:
def plot_grid(df, ax, color_by="depth"):
    # Connect points with same speed across OF
    for speed in all_speeds:
        subset = df[df["expected_RS"] == speed].sort_values("expected_OF")
        ax.plot(subset["PC1"], subset["PC2"], color="#3498db", alpha=0.5, linestyle="-", linewidth=2.0)
        
    # Connect points with same OF across speed
    for of in all_ofs:
        subset = df[df["expected_OF"] == of].sort_values("expected_RS")
        ax.plot(subset["PC1"], subset["PC2"], color="#e74c3c", alpha=0.5, linestyle="--", linewidth=2.0)
        
    # Choose color array based on color_by parameter
    if color_by == "RS":
        c_values = np.log2(df["expected_RS"])
    elif color_by == "OF":
        c_values = np.log2(df["expected_OF"])
    else:
        c_values = df["log_expected_depth"]
        
    # Scatter plot, color-coded accordingly
    scatter = ax.scatter(
        df["PC1"], 
        df["PC2"], 
        c=c_values, 
        cmap="viridis", 
        s=140, 
        edgecolor="black", 
        linewidth=1.0,
        zorder=5
    )
    return scatter


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5))

# 1. Layer 2/3 Pseudopopulation
df_l23, var_l23, n_cells_l23, n_sess_l23 = build_pca_pseudopopulation("Layer 2/3")
plot_grid(df_l23, axes[0])
axes[0].set_xlabel(f"PC1 ({var_l23[0]*100:.1f}%)", fontsize=13)
axes[0].set_ylabel(f"PC2 ({var_l23[1]*100:.1f}%)", fontsize=13)
axes[0].set_title(
    f"Layer 2/3 Manifold (Orthogonal)\n"
    f"{n_sess_l23} sessions ({n_cells_l23} cells)\n"
    f"PC1={var_l23[0]*100:.1f}%, PC2={var_l23[1]*100:.1f}%", 
    fontsize=14, fontweight='bold', pad=10
)
axes[0].grid(True, linestyle=":", alpha=0.5)

# 2. Layer 5 Pseudopopulation
df_l5, var_l5, n_cells_l5, n_sess_l5 = build_pca_pseudopopulation("Layer 5")
sc_l5 = plot_grid(df_l5, axes[1])
axes[1].set_xlabel(f"PC1 ({var_l5[0]*100:.1f}%)", fontsize=13)
axes[1].set_ylabel(f"PC2 ({var_l5[1]*100:.1f}%)", fontsize=13)
axes[1].grid(True, linestyle=":", alpha=0.5)

# 3. Fraction of variance explained vs PC component
pc_indices = np.arange(1, len(var_l23) + 1)
axes[2].plot(pc_indices, var_l23 * 100, 'o-', color='#3498db', label='Layer 2/3', linewidth=2.5, markersize=6)
axes[2].plot(pc_indices, var_l5 * 100, 's-', color='#e74c3c', label='Layer 5', linewidth=2.5, markersize=6)
axes[2].set_xlabel('PC Component', fontsize=13)
axes[2].set_ylabel('Variance Explained (%)', fontsize=13)
axes[2].set_xticks([1, 5, 10, 15, 20, 25, 30])
axes[2].grid(True, linestyle=":", alpha=0.5)
axes[2].legend(frameon=False, fontsize=12)

sns.despine()
plt.tight_layout()
# Add colorbar (only for the first two axes)
cbar = fig.colorbar(sc_l5, ax=axes[:2], pad=0.03, aspect=24, shrink=0.85)
cbar.set_label('Expected Depth (log2 ratio of Speed/Optic Flow)', fontsize=13, fontweight='semibold')

plt.savefig("pca_manifold_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5))

# 1. Layer 2/3 Pseudopopulation
df_l23, var_l23, n_cells_l23, n_sess_l23 = build_pca_pseudopopulation("Layer 2/3")
plot_grid(df_l23, axes[0], color_by='RS')
axes[0].set_xlabel(f"PC1 ({var_l23[0]*100:.1f}%)", fontsize=13)
axes[0].set_ylabel(f"PC2 ({var_l23[1]*100:.1f}%)", fontsize=13)
axes[0].set_title(
    f"Layer 2/3 Manifold (Orthogonal)\n"
    f"{n_sess_l23} sessions ({n_cells_l23} cells)\n"
    f"PC1={var_l23[0]*100:.1f}%, PC2={var_l23[1]*100:.1f}%", 
    fontsize=14, fontweight='bold', pad=10
)
axes[0].grid(True, linestyle=":", alpha=0.5)

# 2. Layer 5 Pseudopopulation
df_l5, var_l5, n_cells_l5, n_sess_l5 = build_pca_pseudopopulation("Layer 5")
sc_l5 = plot_grid(df_l5, axes[1], color_by='RS')
axes[1].set_xlabel(f"PC1 ({var_l5[0]*100:.1f}%)", fontsize=13)
axes[1].set_ylabel(f"PC2 ({var_l5[1]*100:.1f}%)", fontsize=13)
axes[1].grid(True, linestyle=":", alpha=0.5)

# 3. Fraction of variance explained vs PC component
pc_indices = np.arange(1, len(var_l23) + 1)
axes[2].plot(pc_indices, var_l23 * 100, 'o-', color='#3498db', label='Layer 2/3', linewidth=2.5, markersize=6)
axes[2].plot(pc_indices, var_l5 * 100, 's-', color='#e74c3c', label='Layer 5', linewidth=2.5, markersize=6)
axes[2].set_xlabel('PC Component', fontsize=13)
axes[2].set_ylabel('Variance Explained (%)', fontsize=13)
axes[2].set_xticks([1, 5, 10, 15, 20, 25, 30])
axes[2].grid(True, linestyle=":", alpha=0.5)
axes[2].legend(frameon=False, fontsize=12)

sns.despine()
plt.tight_layout()
# Add colorbar (only for the first two axes)
cbar = fig.colorbar(sc_l5, ax=axes[:2], pad=0.03, aspect=24, shrink=0.85)
cbar.set_label('Running Speed (log2 cm/s)', fontsize=13, fontweight='semibold')

plt.savefig("pca_manifold_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5))

# 1. Layer 2/3 Pseudopopulation
df_l23, var_l23, n_cells_l23, n_sess_l23 = build_pca_pseudopopulation("Layer 2/3")
plot_grid(df_l23, axes[0], color_by='OF')
axes[0].set_xlabel(f"PC1 ({var_l23[0]*100:.1f}%)", fontsize=13)
axes[0].set_ylabel(f"PC2 ({var_l23[1]*100:.1f}%)", fontsize=13)
axes[0].set_title(
    f"Layer 2/3 Manifold (Orthogonal)\n"
    f"{n_sess_l23} sessions ({n_cells_l23} cells)\n"
    f"PC1={var_l23[0]*100:.1f}%, PC2={var_l23[1]*100:.1f}%", 
    fontsize=14, fontweight='bold', pad=10
)
axes[0].grid(True, linestyle=":", alpha=0.5)

# 2. Layer 5 Pseudopopulation
df_l5, var_l5, n_cells_l5, n_sess_l5 = build_pca_pseudopopulation("Layer 5")
sc_l5 = plot_grid(df_l5, axes[1], color_by='OF')
axes[1].set_xlabel(f"PC1 ({var_l5[0]*100:.1f}%)", fontsize=13)
axes[1].set_ylabel(f"PC2 ({var_l5[1]*100:.1f}%)", fontsize=13)
axes[1].grid(True, linestyle=":", alpha=0.5)

# 3. Fraction of variance explained vs PC component
pc_indices = np.arange(1, len(var_l23) + 1)
axes[2].plot(pc_indices, var_l23 * 100, 'o-', color='#3498db', label='Layer 2/3', linewidth=2.5, markersize=6)
axes[2].plot(pc_indices, var_l5 * 100, 's-', color='#e74c3c', label='Layer 5', linewidth=2.5, markersize=6)
axes[2].set_xlabel('PC Component', fontsize=13)
axes[2].set_ylabel('Variance Explained (%)', fontsize=13)
axes[2].set_xticks([1, 5, 10, 15, 20, 25, 30])
axes[2].grid(True, linestyle=":", alpha=0.5)
axes[2].legend(frameon=False, fontsize=12)

sns.despine()
plt.tight_layout()
# Add colorbar (only for the first two axes)
cbar = fig.colorbar(sc_l5, ax=axes[:2], pad=0.03, aspect=24, shrink=0.85)
cbar.set_label('Optic Flow (log2 nominal units)', fontsize=13, fontweight='semibold')

plt.savefig("pca_manifold_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## Analysis Summary

Using Pseudopopulation PCA across all aligned sessions allows us to capture the average state-space geometry cleanly without cell-identity correspondence concerns:

1. **Layer 2/3 (Orthogonal):**
   - Variance is shared across both principal components, and the condition averages trace out a clean 2D grid-like manifold.
   - This confirms that Layer 2/3 encodes running speed and optic flow independently/orthogonally.

2. **Layer 5 (Ratio/Depth):**
   - The 2D grid collapses into a 1D trajectory (a single curved line).
   - The log ratio of Speed/Optic Flow (expected depth) is ordered sequentially along this 1D path. This confirms that Layer 5 population encodes depth (the ratio of Speed and Optic Flow) rather than representing them as separate dimensions.